<a href="https://colab.research.google.com/github/Antibote/pinn-eos-aluminum/blob/main/notebooks/section4_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Раздел 4. Инференс, упаковка и сжатие

Порядок: **раздел 0** → 4.1 → 4.2 → 4.3 → 4.4 → 4.5. Нужен обученный `artifacts/pinn.pt` (раздел 3) — он уже в репозитории.

- **4.1** — класс `EOSModel`: `predict`, `sound_speed`, `check_physics` (код — `src/eos/inference.py`);
- **4.2** — размер модели и время на 10⁵ запросов на CPU и GPU;
- **4.3** — экспорт в TorchScript и ONNX, проверка, что ответы совпадают;
- **4.4** — сжатие: float16, int8 только весов, динамическая int8;
- **4.5** — единая таблица baseline / Vanilla / PINN.

## 0. Подготовка (общая, запускать каждому)

In [1]:
import torch
print('torch', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'нет — включите GPU в настройках среды')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch 2.11.0+cu130
GPU: Tesla T4


In [ ]:
# Репозиторий: код (src/eos) и папка artifacts — сюда сохраняются все результаты.
# На GitHub их отправляет save_to_repo('что сохранили') — см. ячейку ниже.
import json, subprocess, sys, urllib.parse, urllib.request, zipfile
from pathlib import Path

REPO = Path('pinn-eos-aluminum')
REPO_URL = 'https://github.com/Antibote/pinn-eos-aluminum.git'
_TOKEN = None


def git(*args):
    """git в папке репозитория. Токен в сообщениях об ошибке скрыт."""
    r = subprocess.run(['git', '-C', str(REPO), *args], capture_output=True, text=True)
    if r.returncode:
        msg = (r.stderr or r.stdout).strip()
        raise RuntimeError(f'git {args[0]}: ' + (msg.replace(_TOKEN, '***') if _TOKEN else msg))
    return r.stdout


if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', REPO_URL, str(REPO)], check=True)
git('config', 'user.name', 'Colab')
git('config', 'user.email', 'colab@users.noreply.github.com')
git('pull', '-q', '--rebase', '--autostash')
sys.path.insert(0, str(REPO / 'src'))

ARTIFACTS = REPO / 'artifacts'
FIGURES = ARTIFACTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
print('Артефакты:', ARTIFACTS.resolve())


In [ ]:
def save_to_repo(message):
    """Коммитит папку artifacts и отправляет её на GitHub.

    Нужен секрет Colab GITHUB_TOKEN (значок ключа слева) с доступом к репозиторию.

    Args:
        message: что сохранили, например 'Бейзлайн: метрики и карта ошибок'.
    """
    global _TOKEN
    from google.colab import userdata
    _TOKEN = userdata.get('GITHUB_TOKEN')

    big = [str(p) for p in ARTIFACTS.rglob('*') if p.is_file() and p.stat().st_size > 50 * 2**20]
    if big:
        raise RuntimeError(f'Файлы больше 50 МБ GitHub не примет: {big}')
    git('add', 'artifacts')
    changed = git('diff', '--cached', '--name-only', '--', 'artifacts').split()
    if not changed:
        print('Нечего сохранять: artifacts не изменилась')
        return

    # Автор коммита — владелец токена
    req = urllib.request.Request('https://api.github.com/user', headers={'Authorization': f'token {_TOKEN}'})
    me = json.load(urllib.request.urlopen(req))
    author = f"{me.get('name') or me['login']} <{me['id']}+{me['login']}@users.noreply.github.com>"

    git('commit', '-q', '-m', message, '--author', author)
    url = REPO_URL.replace('https://', f'https://x-access-token:{_TOKEN}@')
    git('pull', '-q', '--rebase', url, 'main')
    git('push', '-q', url, 'HEAD:main')
    print('На GitHub:', git('log', '-1', '--format=%h %s').strip())
    print('Файлы:', *changed, sep='\n  ')


In [ ]:
# Данные с Яндекс Диска (в репозиторий не попадают)
DATA = REPO / 'data'
if not list(DATA.glob('Obl*.txt')):
    api = 'https://cloud-api.yandex.net/v1/disk/public/resources/download?' + urllib.parse.urlencode(
        {'public_key': 'https://disk.yandex.ru/d/iUSbYrJcsbXSbw', 'path': '/Материалы к хакатону.zip'})
    href = json.load(urllib.request.urlopen(api))['href']
    urllib.request.urlretrieve(href, 'data.zip')
    zipfile.ZipFile('data.zip').extractall(DATA)
print(len(list(DATA.glob('Obl*.txt'))), 'файлов данных в', DATA)


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from eos.data import load_all, region_grid

df = load_all(DATA)
print(df.shape)
df.head()

(1130053, 5)


,rho,T,P,E,region
0,0.001000,0.1,0.000515,-46.797895,Obl01
1,0.001030,0.1,0.000540,-46.790300,Obl01
2,0.001060,0.1,0.000569,-46.782552,Obl01
3,0.001091,0.1,0.000597,-46.774662,Obl01
4,0.001124,0.1,0.000626,-46.766630,Obl01


---
## 4. Инференс, упаковка и сжатие

### 4.1 Класс-обёртка

In [ ]:
# 4.1 Класс-обёртка EOSModel (код — src/eos/inference.py): predict, sound_speed, check_physics
import importlib, json, time, torch
import eos.normalize as nz, eos.model as em, eos.pinn as pn, eos.metrics as mt, eos.baseline as bl, eos.inference as inf
for m in (nz, em, pn, mt, bl, inf):
    importlib.reload(m)
pd.set_option('display.width', 250)

MODEL = ARTIFACTS / 'pinn.pt'
NORM = ARTIFACTS / 'normalization.json'
model = inf.EOSModel(MODEL, NORM, device=DEVICE)                          # float32 — быстрый
model64 = inf.EOSModel(MODEL, NORM, device=DEVICE, dtype=torch.float64)   # float64 — для метрик
print('Конфигурация сети:', model.net.config)

# Примеры: обычный алюминий, сжатый горячий, разреженная плазма
rho_ex = np.array([2.7, 10.0, 0.01, 1000.0])
T_ex = np.array([300.0, 1e4, 1e6, 1e5])
P_ex, e_ex = model.predict(rho_ex, T_ex)
flags = model.check_physics(rho_ex, T_ex)
display(pd.DataFrame({'ρ, г/см³': rho_ex, 'T, K': T_ex, 'P, ГПа': P_ex, 'e, кДж/г': e_ex,
                      'c_s, км/с': model.sound_speed(rho_ex, T_ex),
                      **{k: v for k, v in flags.items() if k != 'cons_residual'}}))

# Проверка физики на 100 000 случайных точках всей области (ρ, T)
rho_q, T_q = bl.random_queries(100_000, seed=1)
flags = model.check_physics(rho_q, T_q)
print('Доля точек, где условие выполнено, %:')
print(pd.Series({k: 100 * v.mean() for k, v in flags.items() if k != 'cons_residual'}).round(3).to_string())

### 4.2 Размер и скорость

In [ ]:
# 4.2 Размер модели и время на 10^5 запросов (CPU и GPU)
size = {'файл pinn.pt, МБ': MODEL.stat().st_size / 2**20,
        'параметров': sum(p.numel() for p in model.net.parameters())}
print(pd.Series(size).to_string(), '\nЛимит по заданию: 10 МБ')

speed = {}
for dev in ['cpu'] + (['cuda'] if torch.cuda.is_available() else []):
    for dt in (torch.float32, torch.float64):
        mm = inf.EOSModel(MODEL, NORM, device=dev, dtype=dt)
        speed[f'PINN {dev} {str(dt)[6:]}'] = bl.benchmark(mm, n=100_000)   # лучшее из 5 прогонов
baseline = bl.BilinearEOS(df)
speed['Бейзлайн (билинейная, CPU)'] = bl.benchmark(baseline, n=100_000)
speed = pd.Series(speed, name='время на 10^5 запросов, с')
display(speed.round(4).to_frame())
with open(ARTIFACTS / 'metrics_speed.json', 'w') as f:
    json.dump({**size, **speed.to_dict()}, f, indent=2, ensure_ascii=False)

### 4.3 Экспорт

In [ ]:
# 4.3 Экспорт: TorchScript и ONNX (нормировка зашита внутрь — вход ρ, T, выход P, e)
import subprocess, sys
try:
    import onnx, onnxruntime
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnx', 'onnxruntime'], check=True)

TS_PATH, ONNX_PATH = ARTIFACTS / 'pinn_torchscript.pt', ARTIFACTS / 'pinn.onnx'
ts = inf.export_torchscript(model.net, TS_PATH)
inf.export_onnx(model.net, ONNX_PATH)
onnx_model = inf.OnnxEOS(ONNX_PATH)

# Совпадают ли ответы экспортированных моделей с исходной (float64)
P0, e0 = model64.predict(rho_q, T_q)
with torch.no_grad():
    P_ts, e_ts = (x.numpy() for x in ts(torch.tensor(rho_q, dtype=torch.float32),
                                         torch.tensor(T_q, dtype=torch.float32)))
P_ox, e_ox = onnx_model.predict(rho_q, T_q)
rel = lambda P, e: (np.max(np.abs(P / P0 - 1)), np.max(np.abs(e - e0) / np.maximum(np.abs(e0), 1)))
export = pd.DataFrame({
    'TorchScript': [TS_PATH.stat().st_size / 2**20, *rel(P_ts, e_ts), np.nan],
    'ONNX': [ONNX_PATH.stat().st_size / 2**20, *rel(P_ox, e_ox), bl.benchmark(onnx_model, n=100_000)],
}, index=['размер, МБ', 'макс. отн. отличие P', 'макс. отличие e (как в метрике)', 'время 10^5 (ONNX Runtime, CPU), с'])
display(export.T)
print('Загрузка без нашего кода: torch.jit.load(path)(rho, T)  или  onnxruntime.InferenceSession(path)')

### 4.4 Сжатие

In [ ]:
# 4.4 Сжатие: float16, int8 (динамическая квантизация), int8 только весов. Точность — на общем test
test_df = df.iloc[np.load(ARTIFACTS / 'split.npz')['test']]

FP16_PATH, INT8W_PATH = ARTIFACTS / 'pinn_fp16.pt', ARTIFACTS / 'pinn_int8w.pt'
inf.save_fp16(model.net, FP16_PATH)
net_w8, packed = inf.compress_int8_weights(model.net)
inf.save_int8_weights(net_w8, packed, INT8W_PATH)
net_q8 = inf.compress_int8(model.net)
q8_buf = ARTIFACTS / 'pinn_int8_dynamic.pt'
torch.save(net_q8.state_dict(), q8_buf)

variants = {
    'PINN float32 (исходная)': (model64, MODEL),
    'PINN float16': (inf.EOSModel(FP16_PATH, NORM, device=DEVICE, dtype=torch.float64), FP16_PATH),
    'PINN int8 (только веса)': (inf.EOSModel.from_net(inf.load_int8_weights(INT8W_PATH, NORM), device=DEVICE,
                                                     dtype=torch.float64), INT8W_PATH),
    'PINN int8 (динамическая, CPU)': (inf.EOSModel.from_net(net_q8), q8_buf),
}
rows = []
for name, (mm, path) in variants.items():
    met, _ = mt.evaluate(mm.predict, test_df, name)
    met['size_MB'] = path.stat().st_size / 2**20
    rows.append(met)
compress = pd.DataFrame(rows).set_index('model')
compress.to_csv(ARTIFACTS / 'metrics_compression.csv')
display(compress[['size_MB', 'err_P_median_%', 'err_P_p99_%', 'err_e_median_%', 'err_e_p99_%',
                  'viol_dP_drho_%', 'viol_de_dT_%']].round(4))
q8_buf.unlink()   # динамическая int8 непригодна (см. таблицу) — в репозиторий не кладём

### 4.5 Таблица сравнения

In [ ]:
# 4.5 Единая таблица сравнения baseline / Vanilla / PINN (+ сжатая PINN) — на одном test
rows = []
for name in ['metrics_baseline.json', 'metrics_vanilla.json']:
    with open(ARTIFACTS / name) as f:
        rows.append(json.load(f))
pinn_met, _ = mt.evaluate(model64.predict, test_df, 'PINN')
pinn_met['size_MB'] = MODEL.stat().st_size / 2**20
fp16_met = compress.loc['PINN float16'].to_dict() | {'model': 'PINN float16'}
gpu = 'PINN cuda float32' if 'PINN cuda float32' in speed else 'PINN cpu float32'
pinn_met['time_1e5_s'] = fp16_met['time_1e5_s'] = speed[gpu]
pinn_met['time_cpu_1e5_s'] = fp16_met['time_cpu_1e5_s'] = speed['PINN cpu float32']
rows += [pinn_met, fp16_met]

final = pd.DataFrame(rows).set_index('model')
base_row = final.index.str.startswith('Baseline')            # бейзлайн считается только на CPU
final.loc[base_row, 'time_cpu_1e5_s'] = final.loc[base_row, 'time_1e5_s']
cols = {'err_P_median_%': 'ошибка P, медиана %', 'err_P_p99_%': 'ошибка P, p99 %',
        'err_e_median_%': 'ошибка e, медиана %', 'err_e_p99_%': 'ошибка e, p99 %',
        'viol_dP_drho_%': '∂P/∂ρ ≤ 0, %', 'viol_de_dT_%': '∂e/∂T < 0, %',
        'size_MB': 'размер, МБ', 'time_1e5_s': 'время 10^5, с (сети — GPU, бейзлайн — CPU)',
        'time_cpu_1e5_s': 'время 10^5 на CPU, с'}
table = final[list(cols)].rename(columns=cols)
table.to_csv(ARTIFACTS / 'metrics_final.csv')
try:
    md = table.to_markdown(floatfmt='.4g')            # таблица для README и отчёта
    (ARTIFACTS / 'metrics_final.md').write_text(md)
except ImportError:
    print('Для metrics_final.md нужен пакет tabulate: pip install tabulate')
display(table.T.round(4))
print('Нарушения у Vanilla посчитаны во float32 (раздел 2) — часть из них шум округления.')
save_to_repo('4: EOSModel, скорость, экспорт TorchScript/ONNX, сжатие, итоговая таблица')